<div dir="rtl" align="right">

# تحليلُ المكوناتِ المستقلةِ \(ICA\)

**مجموعةُ البياناتِ**: PhysioNet Auditory EEG  
**القنواتُ**: P4, Cz, F8, T7  
**معدّلُ أخذِ العيناتِ**: 200 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

يَفترضُ تحليلُ المكوناتِ المستقلةِ أنَّ الإشارةَ مزيجٌ خطّيٌّ من مصادرَ مستقلّة. نَستخدمُ MNE-Python لِفَصلِ القنواتِ الأربعِ إلى مكوناتٍ مستقلّةٍ تَلتقطُ النشاطَ الدماغيَّ والآثارِ الشائبة.

## المُخرجاتُ المُتوقّعةُ

- مخططانِ: الأعلى الإشارةُ الأصليّةُ بأربعِ قنواتٍ، والأسفل أربعُ مكوناتٍ مستقلّةٍ
- كلُّ مكوّنٍ يَلتقطُ نمطاً مختلفاً من النشاطِ أو الآثارِ
- المكوّنُ ذو السعةِ الأعلى قدْ يَكونُ أثراً شائباً

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ | المعنى |
| --- | --- | --- |
| القنواتُ | 4 | P4, Cz, F8, T7 |
| n_components | 4 | عددُ المكوناتِ |
| random_state | 97 | بذرةٌ عشوائيّةٌ |
| max_iter | 800 | الحدُّ الأقصى لِلتكرارِ |

</div>

<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>

In [ ]:
!pip install mne scikit-learn EMD-signal scipy numpy plotly wfdb


<div dir="rtl" align="right">

## 2. استنساخُ المستودعِ وتنزيلُ بياناتِ مُشاركٍ واحدٍ

نَنزّلُ مُشاركًا واحدًا فقط (`--subjects 1`) لتسريعِ التجربةِ في بيئةِ Colab.

</div>

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


<div dir="rtl" align="right">

## 3. تحميلُ إشارةِ EEG

نحمّلُ تسجيلَ المُشاركِ 1 في التجربةِ 1، الجلسةِ 2.

</div>

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
fs = 200

print(f'Channels: {ch_names}')
print(f'Signal length: {len(eeg_data)} samples ({len(eeg_data)/fs:.1f} seconds)')


<div dir="rtl" align="right">

## 4. تطبيقُ تحليلِ المكوناتِ المستقلةِ

نُنشئُ كائنَ `Raw` من MNE ثمّ نُطبّقُ `ICA` بأربعَ مكوناتٍ.

</div>

In [ ]:
import mne

info = mne.create_info(ch_names, sfreq=fs, ch_types='eeg')
raw = mne.io.RawArray(eeg_data.T * 1e-6, info, verbose=False)

ica = mne.preprocessing.ICA(
    n_components=4, random_state=97, max_iter=800, verbose=False
)
ica.fit(raw, verbose=False)
components = ica.get_sources(raw).get_data()
print(f'ICA components shape: {components.shape}')


<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- كلُّ مكوّنٍ يَلتقطُ نمطاً مختلفاً من النشاطِ
- المكوّنُ ذو السعةِ الأعلى قدْ يَكونُ أثراً شائباً
- استخدمْ أداةَ التكبيرِ لِفحصِ نطاقاتٍ زمنيةٍ مُحدّدةٍ


</div>

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

n_plot = min(5000, len(eeg_data))
t_sec = np.arange(n_plot) / fs
colors = ['blue', 'orange', 'green', 'red']

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Original EEG Signal', 'ICA Independent Components'))
for i in range(4):
    offset = i * 200
    fig.add_trace(go.Scatter(x=t_sec, y=eeg_data[:n_plot, i] + offset,
                             name=ch_names[i], line=dict(color=colors[i], width=0.5)),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=t_sec, y=components[i, :n_plot] * 1e6 + offset,
                             name=f'IC{i}', line=dict(color=colors[i], width=0.5)),
                  row=2, col=1)
fig.update_layout(height=700, title_text='ICA Decomposition - Artifact Separation',
                  xaxis2_title='Time (s)', showlegend=True)
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- ICA يَفصلُ الإشارةَ إلى مصادرَ مستقلّةٍ دونَ معرفةٍ مُسبقةٍ
- يَفترضُ استقلاليّةَ المصادرَ وعدمَ غاوسيّتِها
- يَعملُ بشكلٍ أفضلَ معَ عددٍ كبيرٍ من القنواتِ
- المكوّنُ ذو السعةِ الأعلى غالباً ما يَكونُ أثراً شائباً


</div>